# JuaKazi SW Bias Classifier — Fine-tune afro-xlmr-base

Fine-tunes **Davlan/afro-xlmr-base** on the JuaKazi Swahili ground truth dataset for binary gender bias classification.

**Before running:**
1. Set runtime to **GPU** → Runtime → Change runtime type → T4 GPU
2. Upload `ground_truth_sw_v5.csv` when prompted in Step 3
3. Have your HuggingFace token ready (Settings → Access Tokens on huggingface.co)

**Expected time:** ~45 min on T4  
**Expected SW F1 after fine-tuning:** 0.88–0.93

## Step 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu}  ({mem:.1f} GB VRAM)")
else:
    print("No GPU detected — go to Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU required for fine-tuning")

## Step 2 — Install dependencies

In [ ]:
%%capture
!pip install transformers>=4.40.0 datasets accelerate huggingface_hub

## Step 3 — Upload ground truth CSV

Upload `eval/ground_truth_sw_v5.csv` from your local machine.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()   # select ground_truth_sw_v5.csv
csv_path = list(uploaded.keys())[0]
print(f"Uploaded: {csv_path} ({os.path.getsize(csv_path) / 1e6:.1f} MB)")

## Step 4 — Load and inspect dataset

In [ ]:
import csv

def load_dataset(csv_path: str):
    texts, labels = [], []
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            text = (row.get("text") or "").strip()
            has_bias = (row.get("has_bias") or "").lower() == "true"
            bias_label = (row.get("bias_label") or "").lower()
            # 1 = bias (stereotype or derogation), 0 = neutral/counter
            label = 1 if (has_bias or bias_label in ("stereotype", "derogation")) else 0
            if text:
                texts.append(text)
                labels.append(label)
    return texts, labels

texts, labels = load_dataset(csv_path)
n_bias    = sum(labels)
n_neutral = len(labels) - n_bias

print(f"Total rows:  {len(texts):,}")
print(f"bias=1:      {n_bias:,}  ({100*n_bias/len(labels):.1f}%)")
print(f"neutral=0:   {n_neutral:,}  ({100*n_neutral/len(labels):.1f}%)")
print(f"\nSample text: {texts[0][:120]}")

## Step 5 — Tokenize and create train/val splits

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch

MODEL_ID = "Davlan/afro-xlmr-base"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 80/20 split
split = int(len(texts) * 0.8)
train_texts, val_texts   = texts[:split], texts[split:]
train_labels, val_labels = labels[:split], labels[split:]

print(f"Train: {len(train_texts):,}   Val: {len(val_texts):,}")

class BiasDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True, max_length=128
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = BiasDataset(train_texts, train_labels)
val_dataset   = BiasDataset(val_texts,   val_labels)
print("Datasets ready.")

## Step 6 — Load model and configure training

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np

print(f"Loading model: {MODEL_ID}")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model.config.id2label = {0: "NEUTRAL", 1: "BIAS"}
model.config.label2id = {"NEUTRAL": 0, "BIAS": 1}

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {params:.0f}M")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./sw_bias_classifier_v1",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer configured. Ready to train.")

## Step 7 — Train

~45 minutes on T4. Watch the `eval_f1` column — it should climb from ~0.50 to ~0.88 by epoch 3.

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete.")

## Step 8 — Evaluate on validation set

In [ ]:
metrics = trainer.evaluate()
print("\n=== Validation metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## Step 9 — Save model locally

In [ ]:
OUTPUT_DIR = "./sw_bias_classifier_v1_final"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

import os
size_mb = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
) / 1e6
print(f"Total size: {size_mb:.0f} MB")

## Step 10 — Push to HuggingFace Hub

Get your token from https://huggingface.co/settings/tokens (write access needed).  
The model will be uploaded as `juakazike/sw-bias-classifier-v1`.

In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted

In [ ]:
HF_REPO = "juakazike/sw-bias-classifier-v1"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## Step 11 — Download model zip (optional)

If you want to run locally without re-downloading from HF:

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/sw_bias_classifier_v1_final"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(zip_path + ".zip")
print("Download started.")

## Next steps — activate in JuaKazi

After pushing to HF, set the env var in your deployment:

```bash
export JUAKAZI_ML_MODEL=juakazike/sw-bias-classifier-v1
export JUAKAZI_ML_THRESHOLD=0.75
```

Or in HuggingFace Spaces → Settings → Repository secrets.

The JuaKazi `BiasDetector` will automatically use the fine-tuned model as Stage 2 fallback when `JUAKAZI_ML_MODEL` is set.

**Verify locally:**
```python
from eval.ml_classifier import classify, model_id
print(model_id())          # should show juakazike/sw-bias-classifier-v1
from eval.models import Language
score = classify("Daktari wa kiume alifika", Language.SWAHILI)
print(f"Bias score: {score:.3f}")   # should be > 0.75
```